In [2]:
%pip install jieba

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import zipfile
import requests
import jieba
import pandas as pd

# =========================
# 1. 自动下载函数
# =========================

def download_file(url, filename):
    """如果本地没有文件，就从网上自动下载"""
    if os.path.exists(filename):
        print("已存在，跳过下载：", filename)
        return
    print("正在下载：", filename)
    response = requests.get(
        url,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()
    with open(filename, "wb") as f:
        f.write(response.content)
    print("下载完成：", filename)

# =========================
# 2. 下载 BosonNLP 情感词典
# =========================
boson_url = (
    "https://github.com/nju-zw/"
    "Sentiment-Analysis-of-Online-Health-Community/"
    "raw/refs/heads/main/BosonNLP/"
    "BosonNLP_sentiment_score.zip"
)
boson_zip = "BosonNLP_sentiment_score.zip"
download_file(
    boson_url,
    boson_zip
)
# 解压
boson_folder = "boson_dict"
if not os.path.exists(boson_folder):
    os.makedirs(boson_folder)
    with zipfile.ZipFile(boson_zip, "r") as z:
        z.extractall(boson_folder)
    print("词典解压完成")

# 自动寻找 txt 文件
dict_file = None
for root, dirs, files in os.walk(boson_folder):
    for file in files:
        if file.endswith(".txt") and "sentiment_score" in file:
            dict_file = os.path.join(root, file)
            break
print("情感词典：", dict_file)

# =========================
# 3. 读取情感词典
# =========================

sentiment_dict = {}
with open(
    dict_file,
    "r",
    encoding="utf-8"
) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 2:
            word = parts[0]
            try:
                score = float(parts[1])
                sentiment_dict[word] = score
            except:
                pass
print(
    "情感词数量：",
    len(sentiment_dict)
)

# =========================
# 4. 情感分析函数
# =========================
def sentiment_analysis(text):
    words = jieba.lcut(str(text))
    score = 0
    emotion_words = []
    for word in words:
        if word in sentiment_dict:
            word_score = sentiment_dict[word]
            score += word_score
            emotion_words.append(
                (word, round(word_score, 2))
            )

    if score > 0:
        result = "正面"
    elif score < 0:
        result = "负面"
    else:
        result = "中性"
    return round(score, 2), result, emotion_words

# =========================
# 5. 自己输入一句话测试
# =========================

text = "这个酒店环境很好，服务也很热情，但是价格有点贵。"
score, result, words = sentiment_analysis(text)
print("\n原文：")
print(text)
print("\n识别出的情感词：")
print(words)

print("\n情感得分：")
print(score)

print("\n情感判断：")
print(result)

# =========================
# 6. 自动下载 ChnSentiCorp 语料
# =========================

corpus_url = (
    "https://raw.githubusercontent.com/"
    "SophonPlus/ChineseNlpCorpus/master/"
    "datasets/ChnSentiCorp_htl_all/"
    "ChnSentiCorp_htl_all.csv"
)
corpus_file = "ChnSentiCorp_htl_all.csv"
download_file(
    corpus_url,
    corpus_file
)

# =========================
# 7. 读取语料
# =========================

data = pd.read_csv(corpus_file)
data = data.dropna(
    subset=["review"]
)
print("\n语料总数：", len(data))
print(data.head())

# =========================
# 8a. 从真实语料随机抽5条
# =========================

sample = data.sample(
    5,
    random_state=4
)

for index, row in sample.iterrows():
    text = row["review"]
    real_label = row["label"]
    score, result, words = sentiment_analysis(text)
    if real_label == 1:
        real_result = "正面"
    else:
        real_result = "负面"

    print("\n==========================")

    print("评论：")
    print(text[:150])

    print("\n情感词：")
    print(words[:10])

    print("\n词典得分：", score)

    print("词典预测：", result)

    print("真实标签：", real_result)

C:\AppData\Local\Programs\Python\Python312\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Loading model from cache C:\AppData\Local\Temp\jieba.cache


已存在，跳过下载： BosonNLP_sentiment_score.zip
情感词典： boson_dict\BosonNLP_sentiment_score.txt
情感词数量： 114766


Loading model cost 0.638 seconds.
Prefix dict has been built successfully.



原文：
这个酒店环境很好，服务也很热情，但是价格有点贵。

识别出的情感词：
[('这个', -0.29), ('酒店', 0.35), ('环境', 0.73), ('很', 0.53), ('好', 2.07), ('，', -0.02), ('服务', 1.13), ('很', 0.53), ('热情', 1.7), ('，', -0.02), ('但是', 0.04), ('价格', 1.84), ('有点', 0.46), ('贵', -1.55), ('。', -0.1)]

情感得分：
7.4

情感判断：
正面
已存在，跳过下载： ChnSentiCorp_htl_all.csv

语料总数： 7765
   label                                             review
0      1  距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较...
1      1                       商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2      1         早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3      1  宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4      1               CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风

评论：
去柳州还算不错的选择。周围挺热闹的，书店，步行街...服务很好，房间也还可以。就是卫生间太小了，还有早餐太一般了。反正去柳州算是个OK。宾馆反馈2008年7月31日：您的意见我们已向西餐厅做了反馈，自助早餐的品种已做了调整，相信能让您满意。柳州宾馆地处柳州市的商业中心，吃，住，行，游，购，娱十分便利

情感词：
[('去', -0.32), ('柳州', 0.16), ('还', -0.69), ('算', -1.06), ('不错', 2.65), ('的', 0.04), ('选择', 0.8), ('。', -0.1), ('周围', -0.49), ('挺', 0.41)]

词典得分： 40.96
词典预测： 正面
真实标

In [4]:
# =========================
# 8b. 对全部语料进行情感分析
# =========================

import os
from IPython.display import display, Markdown


# 保存所有分析结果
results = []


# 对全部评论逐条分析
for i, row in data.iterrows():

    text = str(row["review"])
    real_label = int(row["label"])

    # 调用前面已经定义好的情感分析函数
    score, prediction, emotion_words = sentiment_analysis(text)

    # 把真实标签转换成中文
    if real_label == 1:
        real_result = "正面"
    else:
        real_result = "负面"

    # 保存结果
    results.append({
        "编号": i + 1,
        "真实值": real_result,
        "预测值": prediction,
        "计算的实际值": score
    })


# =========================
# 9. 转换成 DataFrame
# =========================

result_df = pd.DataFrame(results)


# =========================
# 10. Markdown显示前10条
# =========================

print("全部语料数量：", len(result_df))

display(
    Markdown(
        "## 情感分析结果（前10条）\n\n"
        + result_df.head(10).to_markdown(index=False)
    )
)


# =========================
# 11. 计算预测正确率
# =========================

# 中性预测因为真实语料只有正面/负面，
# 所以中性也计算为预测错误
result_df["是否正确"] = (
    result_df["真实值"] == result_df["预测值"]
)

correct_count = result_df["是否正确"].sum()

total_count = len(result_df)

accuracy = correct_count / total_count


# 用 Markdown 显示整体结果
summary_md = f"""
## 全部语料分析结果

- **语料总数：** {total_count}
- **预测正确：** {correct_count}
- **预测错误：** {total_count - correct_count}
- **准确率：** {accuracy:.2%}

### 预测结果分布

- **预测为正面：** {(result_df["预测值"] == "正面").sum()}
- **预测为负面：** {(result_df["预测值"] == "负面").sum()}
- **预测为中性：** {(result_df["预测值"] == "中性").sum()}
"""

display(Markdown(summary_md))


# =========================
# 12. 保存全部结果到 CSV
# =========================

# 创建 C:\temp 文件夹
output_folder = r"C:\temp"

os.makedirs(
    output_folder,
    exist_ok=True
)


# CSV文件名
output_file = os.path.join(
    output_folder,
    "sentiment_results.csv"
)


# 按照要求只输出4列
csv_result = result_df[
    [
        "编号",
        "真实值",
        "预测值",
        "计算的实际值"
    ]
]


# utf-8-sig 可以避免Windows Excel打开中文乱码
csv_result.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print("\n分析完成！")
print("结果文件已经保存到：")
print(output_file)

全部语料数量： 7765


## 情感分析结果（前10条）

|   编号 | 真实值   | 预测值   |   计算的实际值 |
|-------:|:---------|:---------|---------------:|
|      1 | 正面     | 正面     |           0.46 |
|      2 | 正面     | 正面     |           9.99 |
|      3 | 正面     | 正面     |           0.74 |
|      4 | 正面     | 正面     |          38.9  |
|      5 | 正面     | 负面     |          -1.84 |
|      6 | 正面     | 正面     |           7.31 |
|      7 | 正面     | 正面     |           7.44 |
|      8 | 正面     | 正面     |           8.01 |
|      9 | 正面     | 正面     |          50.78 |
|     10 | 正面     | 正面     |          12.75 |


## 全部语料分析结果

- **语料总数：** 7765
- **预测正确：** 5919
- **预测错误：** 1846
- **准确率：** 76.23%

### 预测结果分布

- **预测为正面：** 6215
- **预测为负面：** 1548
- **预测为中性：** 2



分析完成！
结果文件已经保存到：
C:\temp\sentiment_results.csv


In [5]:
# =========================
# 13. 整体统计
# =========================

# 判断预测是否正确
result_df["是否正确"] = (
    result_df["真实值"] == result_df["预测值"]
)


# ---------- 基本统计 ----------

total_count = len(result_df)

correct_count = result_df["是否正确"].sum()

wrong_count = total_count - correct_count

accuracy = correct_count / total_count


# ---------- 真实标签统计 ----------

real_positive = (
    result_df["真实值"] == "正面"
).sum()

real_negative = (
    result_df["真实值"] == "负面"
).sum()


# ---------- 预测结果统计 ----------

predict_positive = (
    result_df["预测值"] == "正面"
).sum()

predict_negative = (
    result_df["预测值"] == "负面"
).sum()

predict_neutral = (
    result_df["预测值"] == "中性"
).sum()


# =========================
# Markdown输出整体统计
# =========================

summary_md = f"""
# 情感分析整体统计

## 1. 数据基本情况

| 项目 | 数量 |
|---|---:|
| 全部语料 | {total_count} |
| 真实正面 | {real_positive} |
| 真实负面 | {real_negative} |

---

## 2. 词典法预测结果

| 预测结果 | 数量 |
|---|---:|
| 正面 | {predict_positive} |
| 负面 | {predict_negative} |
| 中性 | {predict_neutral} |

---

## 3. 预测效果

| 指标 | 结果 |
|---|---:|
| 预测正确 | {correct_count} |
| 预测错误 | {wrong_count} |
| 准确率 | {accuracy:.2%} |

"""

display(
    Markdown(summary_md)
)


# 情感分析整体统计

## 1. 数据基本情况

| 项目 | 数量 |
|---|---:|
| 全部语料 | 7765 |
| 真实正面 | 5322 |
| 真实负面 | 2443 |

---

## 2. 词典法预测结果

| 预测结果 | 数量 |
|---|---:|
| 正面 | 6215 |
| 负面 | 1548 |
| 中性 | 2 |

---

## 3. 预测效果

| 指标 | 结果 |
|---|---:|
| 预测正确 | 5919 |
| 预测错误 | 1846 |
| 准确率 | 76.23% |

